## Prediction of Customer Churn Pbobability

**Version 1.0**

The notebook in fourth entry to competition I started with Basics to start with using Random Fore7r, before I test pipelione and ensemble

**Version 2.0**

Random Forests tend to struggle with this because they average predictions from many trees, which often pushes probabilities away from 0 and 1 toward the center.
This version introduces Random Forest Calibration By applying Platt Scaling (Sigmoid calibration), turns a mathematical output into a reliable financial metric for churn prevention strategies.

**Version 3.0**

Introduced GridSearchCV: It no longer guesses the number of trees; it tests 50, 100, 150, 200, 250 and 300 to find the most accurate estimator for specific data.

**Version 4.0**

Introduced notebook pipeline using Balanced Random Forest (from imblearn) designed to:
- Handle churn imbalance properly
- Increase churn probability sensitivity
- Optimize for ROC-AUC
- Improve recall for churners
- Provide calibrated probabilities
- Use stratified CV
- Automatically tune threshold

**Version 5.0***

Added multi-seed ensemble combining XGBoost + Random Forest, optimized for churn imbalance and probability separation.
- XGBoost with scale_pos_weight
- RandomForest with class_weight='balanced'
- 3 seeds
- 5-fold stratified CV
- Rank normalization blending
- Weight optimization
- Threshold tuning for recall

**Version 6.0**

Added CatBoost to XGBoost and RF
  3-model multi-seed ensemble:
- XGBoost
- Random Forest
- CatBoost
- 3 seeds
- 5-fold stratified CV
- Rank blending
- Weight optimization
- Threshold tuning

**Version 7.0**


Added LightGBM to the previous XGBoost + Random Forest + CatBoost multi-seed ensemble to evaluate 4 model ensemble, these four specific algorithms leverage different mathematical "perspectives" on your data:
- XGBoost & LightGBM: These are "Gradient Boosted Decision Trees" (GBDT). They build trees sequentially, where each tree tries to correct the errors of the previous one. They are highly aggressive and excellent at finding complex patterns.
- CatBoost: It uses a specialized technique called "Ordered Boosting" which is specifically designed to reduce bias when dealing with categorical variables. It often finds insights that XGB/LGB miss.
- Random Forest: This is a "Bagging" model. Unlike the others, it builds many independent trees in parallel and averages them. It is much harder to overfit, making it a perfect "anchor" for your ensemble to prevent the boosting models from becoming too volatile.

**Version 8.0**

Added Feature Engineering along with automatic ensemble weight calculation, the model now has
- Feature Engineering (tenure ratios, charge ratios, service counts)
- Automatic Ensemble Weight Search
- Multi-seed ensemble (XGB + LGBM + CatBoost + RF)
- Rank blending
- Grid search for optimal ensemble weights

**Version 9.0**

Improved pipeline by adding
- Automatic feature generation
- Multi-seed base models (XGB + LGBM + CatBoost + RF)
- Out-of-fold predictions
- Stacking meta-model (Logistic Regression)
- Rank normalization

The pipeline can be depicted:

- Raw Data
- Feature Engineering
- Base Models (Level-1)

--   ├── XGBoost  
--   ├── LightGBM  
--   ├── CatBoost
--   └── RandomForest

- OOF Predictions
- Stacking Dataset
- Meta Model (Level-2)
- Logistic Regression
- Final Predictions

**Version 10.0**
 - Raw Data
 - Feature Engineering
 - Encoding + Scaling
 - Level-1 Models
   -- ── XGBoost
   -- ── LightGBM
   -- ── CatBoost
 - OOF Predictions
 - Pseudo-Labeling
 - Level-2 Stacking
   -- ── Ridge
   -- ── Logistic
   -- ── Neural Network
 - Final Ensemble

**Version 11.0**
 - Raw Data
 - Feature Engineering
 - Encoding + Scaling
 - Level-1 Multi Seed Models 
   -- ── XGBoost
   -- ── LightGBM
   -- ── CatBoost
 - OOF Predictions
 - Pseudo-Labeling
 - Level-2 Stacking
   -- ── Ridge
   -- ── Logistic
   -- ── Neural Network
 - Final Ensemble

**Version 12.0**

Added auto ensemble for level 2 stacking

In [1]:
# ============================================================
# 1. Imports
# ============================================================
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import optuna

import warnings

# Suppress the specific Pandas FutureWarning
warnings.filterwarnings("ignore", category=FutureWarning, message=".*Setting an item of incompatible dtype.*")

# Suppress the Scikit-learn UserWarning about feature names
warnings.filterwarnings("ignore", category=UserWarning, message=".*X does not have valid feature names.*")

optuna.logging.set_verbosity(optuna.logging.WARNING)

print("\n Completed loading Libraries")


 Completed loading Libraries


In [2]:
# ============================================================
# 2. Configuration & Data Loading
# ============================================================
N_SPLITS = 5
SEEDS = [42, 100, 2024, 777, 999]
TARGET = "Churn"
ID_COL = "id"

train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")
test_ids = test[ID_COL]
# Map target early
train[TARGET] = train[TARGET].map({"No": 0, "Yes": 1})

print("\n Completed loading data ... .. .")


 Completed loading data ... .. .


In [3]:
# ============================================================
# 3. Enhanced Feature Engineering
# ============================================================

def enhanced_feature_engineering(df):
    # Base ratios
    df["avg_monthly_charge"] = df["TotalCharges"] / (df["tenure"] + 1)
    df["charge_ratio"] = df["MonthlyCharges"] / (df["TotalCharges"] + 1)
    
    # Service interaction features
    service_cols = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 
                    'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
    
    # Count total active services
    df['total_services'] = df[service_cols].apply(lambda x: x.str.contains('Yes|Fiber|DSL').sum(), axis=1)
    
    # Financial stress indicators
    df['high_cost_short_tenure'] = (df['MonthlyCharges'] > df['MonthlyCharges'].median()) & (df['tenure'] < df['tenure'].median())
    df['high_cost_short_tenure'] = df['high_cost_short_tenure'].astype(int)

    return df

train = enhanced_feature_engineering(train)
test = enhanced_feature_engineering(test)

le = LabelEncoder()
train[TARGET] = le.fit_transform(train[TARGET])

print("\n Completed Computing Feature Engineering ... .. .")


 Completed Computing Feature Engineering ... .. .


In [4]:
# ============================================================
# 4. Preprocessing
# ============================================================

# Encode categorical
for col in train.columns:
    if train[col].dtype == 'object' and col not in [ID_COL]:
        full = pd.concat([train[col], test[col]])
        enc = LabelEncoder()
        full_enc = enc.fit_transform(full.astype(str))
        train[col] = full_enc[:len(train)]
        test[col] = full_enc[len(train):]

X = train.drop([TARGET, ID_COL], axis=1)
y = train[TARGET]
X_test = test.drop([ID_COL], axis=1)

X['num_zero'] = (X == 0).sum(axis=1)
X['num_mean'] = X.mean(axis=1)
X['num_std'] = X.std(axis=1)

X_test['num_zero'] = (X_test == 0).sum(axis=1)
X_test['num_mean'] = X_test.mean(axis=1)
X_test['num_std'] = X_test.std(axis=1)

print("\n Completed Preprocessing and Encoding ... .. .")


 Completed Preprocessing and Encoding ... .. .


In [5]:
# ============================================================
# 5. Define Base Model
# ============================================================
def get_models(seed):
    return {
        "xgb": xgb.XGBClassifier(
            n_estimators=8000,
            max_depth=6,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='logloss',
            random_state=seed
        ),
        "lgb": lgb.LGBMClassifier(
            n_estimators=8000,
            learning_rate=0.03,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=seed,
            force_row_wise="true",
            verbosity=-1
        ),
        "cat": cb.CatBoostClassifier(
            iterations=8000,
            learning_rate=0.03,
            depth=6,
            verbose=0,
            random_seed=seed
        ),
    }

print("\n Completed Defining Model Parameters ... .. .")


 Completed Defining Model Parameters ... .. .


In [6]:
# ============================================================
# OOF STORAGE
# ============================================================
oof_preds = {}
test_preds = {}

for model_name in ["xgb", "lgb", "cat"]:
    oof_preds[model_name] = np.zeros(len(X))
    test_preds[model_name] = np.zeros(len(X_test))

print("\n Completed OOF Storage ... .. .")


 Completed OOF Storage ... .. .


In [7]:
# ============================================================
# TRAINING LOOP (MULTI-SEED + CV)
# ============================================================

for seed in SEEDS:
    print(f"\n==== SEED {seed} ====")

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    models = get_models(seed)

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
        print(f"\n--- Fold {fold} ---")

        X_tr, X_val = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[valid_idx]

        for name, model in models.items():
            model.fit(X_tr, y_tr)

            val_pred = model.predict_proba(X_val)[:, 1]
            auc = roc_auc_score(y_val, val_pred)

            print(f"Model: {name} | Seed: {seed} | Fold: {fold} | AUC: {auc:.5f}")

            oof_preds[name][valid_idx] += val_pred / len(SEEDS)
            test_preds[name] += model.predict_proba(X_test)[:, 1] / (len(SEEDS) * N_SPLITS)



==== SEED 42 ====

--- Fold 0 ---
Model: xgb | Seed: 42 | Fold: 0 | AUC: 0.91342
Model: lgb | Seed: 42 | Fold: 0 | AUC: 0.91450
Model: cat | Seed: 42 | Fold: 0 | AUC: 0.91594

--- Fold 1 ---
Model: xgb | Seed: 42 | Fold: 1 | AUC: 0.91447
Model: lgb | Seed: 42 | Fold: 1 | AUC: 0.91562
Model: cat | Seed: 42 | Fold: 1 | AUC: 0.91714

--- Fold 2 ---
Model: xgb | Seed: 42 | Fold: 2 | AUC: 0.91385
Model: lgb | Seed: 42 | Fold: 2 | AUC: 0.91507
Model: cat | Seed: 42 | Fold: 2 | AUC: 0.91643

--- Fold 3 ---
Model: xgb | Seed: 42 | Fold: 3 | AUC: 0.91490
Model: lgb | Seed: 42 | Fold: 3 | AUC: 0.91617
Model: cat | Seed: 42 | Fold: 3 | AUC: 0.91748

--- Fold 4 ---
Model: xgb | Seed: 42 | Fold: 4 | AUC: 0.91213
Model: lgb | Seed: 42 | Fold: 4 | AUC: 0.91333
Model: cat | Seed: 42 | Fold: 4 | AUC: 0.91460

==== SEED 100 ====

--- Fold 0 ---
Model: xgb | Seed: 100 | Fold: 0 | AUC: 0.91450
Model: lgb | Seed: 100 | Fold: 0 | AUC: 0.91574
Model: cat | Seed: 100 | Fold: 0 | AUC: 0.91698

--- Fold 1 ---


In [8]:
# ============================================================
# LEVEL 2 DATASET
# ============================================================

X_l2 = pd.DataFrame(oof_preds)
X_test_l2 = pd.DataFrame(test_preds)

# ============================================================
# AUTO ENSEMBLE (OPTUNA)
# ============================================================
def objective(trial):
    w = np.array([
        trial.suggest_float(f"w_{i}", 0, 1) for i in range(X_l2.shape[1])
    ])
    w = w / w.sum()

    pred = np.dot(X_l2.values, w)
    return roc_auc_score(y, pred)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

best_w = np.array([study.best_params[f"w_{i}"] for i in range(X_l2.shape[1])])
best_w = best_w / best_w.sum()

print("\n Completed level 2 Optuna Tuning ... .. .")




 Completed level 2 Optuna Tuning ... .. .


In [9]:
# ============================================================
# STACKING META MODELS
# ============================================================
meta_models = {
    "ridge": RidgeClassifier(),
    "logistic": LogisticRegression(max_iter=1000),
    "mlp": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500)
}

meta_preds = []

for name, model in meta_models.items():
    model.fit(X_l2, y)
    pred = model.predict_proba(X_test_l2)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_test_l2)
    meta_preds.append(pred)

meta_preds = np.mean(meta_preds, axis=0)

print("\n Completed stacking meta model ... .. .")


 Completed stacking meta model ... .. .


In [10]:
# ============================================================
# FINAL BLEND
# ============================================================

def final_blend_objective(trial):
    w1 = trial.suggest_float("w_weighted", 0, 1)
    w2 = trial.suggest_float("w_meta", 0, 1)

    # normalize
    w_sum = w1 + w2
    w1 /= w_sum
    w2 /= w_sum

    weighted_oof = np.dot(X_l2.values, best_w)

    # train meta models for OOF
    meta_oof_preds = []
    for name, model in meta_models.items():
        model.fit(X_l2, y)
        if hasattr(model, "predict_proba"):
            pred = model.predict_proba(X_l2)[:, 1]
        else:
            pred = model.decision_function(X_l2)
        meta_oof_preds.append(pred)

    meta_oof = np.mean(meta_oof_preds, axis=0)

    final_oof = w1 * weighted_oof + w2 * meta_oof

    return roc_auc_score(y, final_oof)

# Optimize blend
blend_study = optuna.create_study(direction="maximize")
blend_study.optimize(final_blend_objective, n_trials=30)

bw1 = blend_study.best_params["w_weighted"]
bw2 = blend_study.best_params["w_meta"]

# normalize
bw_sum = bw1 + bw2
bw1 /= bw_sum
bw2 /= bw_sum

print("Best Blend Weights:", bw1, bw2)

weighted_pred = np.dot(X_test_l2.values, best_w)
final_pred = bw1 * weighted_pred + bw2 * meta_preds

Best Blend Weights: 0.5220313292027291 0.47796867079727096


In [11]:
# ============================================================
# 8. Submission
# ============================================================
submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: final_pred
})

submission.to_csv("submission.csv", index=False)

print("Submission file created!")

submission.head(20)




Submission file created!


,id,Churn
0,594194,-0.076473
1,594195,-0.152038
2,594196,-0.037236
3,594197,-0.150046
4,594198,0.444890
5,594199,0.051145
6,594200,0.893309
7,594201,-0.151012
8,594202,-0.123729
9,594203,0.219879


In [12]:
# ============================================================
# DIAGNOSTICS
# ============================================================

print("\nBest Weights:")
print(best_w)

print("OOF AUC:", roc_auc_score(y, np.dot(X_l2.values, best_w)))


Best Weights:
[0.00158294 0.20893433 0.78948273]
OOF AUC: 0.9170431971906401
